# MLP (Pipeline B)

**Model**: `MLPRegressor` / **Target**: gdpc1  
**Variables**: Cat3 (53 + COVID = 56) / **n_lags**: 4  
**Scaling**: YES (gradient conditioning) / **HP tuning**: YES  
**10-model averaging**: YES  
**Note**: 212 features for ~194 obs. Alpha grid includes 1e-2 for stronger L2 reg.

In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import norm
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [15, 10]
sys.path.insert(0, os.path.join(os.path.pardir, 'data'))
from helpers import (gen_lagged_data, flatten_data, mean_fill_dataset,
    get_features, split_for_scaler, load_data, PREDICTIONS_DIR, TARGET, COVID)
SEED = 42; np.random.seed(SEED)
TRAIN_START = '1959-01-01'; TRAIN_END = '2007-12-31'; VAL_END = '2016-12-31'
TEST_START = '2017-01-01'; TEST_END = '2025-12-31'
N_LAGS = 4; N_MODELS = 10; VINTAGES = {'m1': -2, 'm2': -1, 'm3': 0}
print('MLP — Cat3, n_lags=4, scaling=YES')

MLP — Cat3, n_lags=4, scaling=YES


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
scaled_cols, unscaled_cols = split_for_scaler(features)
# Phase A: HP tune alpha on validation
tune = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= VAL_END)].reset_index(drop=True)
tune_f = mean_fill_dataset(tune, tune)
tune_fl = flatten_data(tune_f, TARGET, N_LAGS)
tune_fl = tune_fl.loc[tune_fl.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how='any').reset_index(drop=True)
fcols = [c for c in tune_fl.columns if c != 'date' and c != TARGET]
sc_c = [c for c in fcols if c in scaled_cols or any(c.startswith(s + '_') for s in scaled_cols)]
us_c = [c for c in fcols if c in unscaled_cols or any(c.startswith(u + '_') for u in unscaled_cols)]
tscv = TimeSeriesSplit(n_splits=5)
best_score = np.inf; best_alpha = 0.001; best_hls = (64,)
for hls in [(64,), (128,), (64, 64)]:
    for alpha in [1e-4, 1e-3, 1e-2]:
        scores = []
        for tr, va in tscv.split(tune_fl):
            sc = StandardScaler()
            X_tr_s = sc.fit_transform(tune_fl[sc_c].values[tr])
            X_tr = np.hstack([X_tr_s, tune_fl[us_c].values[tr]]) if us_c else X_tr_s
            X_va_s = sc.transform(tune_fl[sc_c].values[va])
            X_va = np.hstack([X_va_s, tune_fl[us_c].values[va]]) if us_c else X_va_s
            m = MLPRegressor(hidden_layer_sizes=hls, alpha=alpha, random_state=SEED, max_iter=300)
            m.fit(X_tr, tune_fl[TARGET].values[tr])
            p = m.predict(X_va)
            scores.append(np.mean((p - tune_fl[TARGET].values[va])**2))
        if np.mean(scores) < best_score:
            best_score = np.mean(scores)
            best_alpha = alpha
            best_hls = hls
print(f'Best MLP params: alpha={best_alpha}, hidden_layer_sizes={best_hls}')

Best MLP params: alpha=0.01, hidden_layer_sizes=(64, 64)


In [3]:
test_quarters = monthly[(monthly['date'] >= TEST_START) & (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3,6,9,12])]['date'].tolist()
predictions = {v: [] for v in VINTAGES}; actuals_list = []; failed = 0
for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly['date'] == pd_q][TARGET].values[0])
    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        train = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= fc)].reset_index(drop=True)
        train_m = gen_lagged_data(metadata, train.copy(), fc, lag=0)
        train_f = mean_fill_dataset(train_m, train_m)
        train_fl = flatten_data(train_f, TARGET, N_LAGS)
        train_fl = train_fl.loc[train_fl.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how='any').reset_index(drop=True)
        if len(train_fl) < 10:
            predictions[vn].append(np.nan); continue
        try:
            sc = StandardScaler()
            X_tr_s = sc.fit_transform(train_fl[sc_c].values)
            X_tr = np.hstack([X_tr_s, train_fl[us_c].values]) if us_c else X_tr_s
            models = []
            for k in range(N_MODELS):
                m = MLPRegressor(hidden_layer_sizes=best_hls, activation='relu', solver='adam',
                    alpha=best_alpha, learning_rate_init=0.001, learning_rate='constant',
                    random_state=SEED+k, max_iter=500)
                m.fit(X_tr, train_fl[TARGET].values); models.append(m)
            tr = monthly[monthly['date'] == fc].reset_index(drop=True)
            tr_m = gen_lagged_data(metadata, tr.copy(), fc, lag=0)
            tr_f = mean_fill_dataset(train_m, tr_m)
            ctx = train_f.tail(N_LAGS + 1).iloc[:-1].copy()
            cmb = pd.concat([ctx, tr_f], ignore_index=True)
            cmb_fl = flatten_data(cmb, TARGET, N_LAGS)
            tr_fl = cmb_fl.tail(1)
            X_te_s = sc.transform(tr_fl[sc_c].values)
            X_te = np.hstack([X_te_s, tr_fl[us_c].values]) if us_c else X_te_s
            preds = [m.predict(X_te)[0] for m in models]
            predictions[vn].append(np.nanmean(preds))
        except Exception:
            predictions[vn].append(np.nan); failed += 1
    if (i+1) % 6 == 0:
        print(f'  {i+1}/{len(test_quarters)}')
print(f'Done. {failed} failures.')

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

...

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\model_notebooks\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a futur

  36/36
Done. 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({'date': test_quarters, 'actual': actuals_list, 'prediction': predictions[vn]}).to_csv(
        os.path.join(PREDICTIONS_DIR, f'mlp_{vn}.csv'), index=False)
    print(f'Saved mlp_{vn}.csv')

Saved mlp_m1.csv


Saved mlp_m2.csv
Saved mlp_m3.csv


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum() > 0 else np.nan
def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum() > 0 else np.nan

panels = {
    'Pre-COVID  (2017-2019)': ('2017-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2017-2025)': ('2017-01-01', '2025-12-31'),
}
a = np.array(actuals_list); d = pd.to_datetime(test_quarters)
print(f'MLP best params: alpha={best_alpha}, hls={best_hls}')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print(f'--- {pn} ---')
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print(f'  {vn}  RMSFE={rmse(a[m], pp[m]):.6f}  MAE={mae(a[m], pp[m]):.6f}')

MLP best params: alpha=0.01, hls=(64, 64)
--- Pre-COVID  (2017-2019) ---
  m1  RMSFE=0.089184  MAE=0.069375
  m2  RMSFE=0.067199  MAE=0.057264
  m3  RMSFE=0.006199  MAE=0.005017
--- COVID      (2020-2021) ---
  m1  RMSFE=0.137236  MAE=0.118168
  m2  RMSFE=0.295160  MAE=0.197478
  m3  RMSFE=0.039702  MAE=0.025269
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.073588  MAE=0.060977
  m2  RMSFE=0.086368  MAE=0.062526
  m3  RMSFE=0.003779  MAE=0.002931
--- Full test  (2017-2025) ---
  m1  RMSFE=0.093451  MAE=0.073806
  m2  RMSFE=0.147517  MAE=0.085421
  m3  RMSFE=0.018563  MAE=0.008614
